# 6.1 Time averaging and relative intensity

**Why can the instantaneous field change while the detector signal stays the same?**

 Relative $|U|^2$ uses a fixed-medium reference. It does not compare absolute irradiance across media. For electric-field amplitudes, irradiance also depends on refractive index, and normal power depends on direction. These plots do not apply that cross-medium calibration.

[Chapter 5](5_0_Grating_Diffraction_in_Three_Dimensions.ipynb) established order directions and collection. Here we isolate one propagating order and model a broad part of its beam as a uniform plane wave, ignoring edges. Its direction is fixed and amplitude is an input; calculating the amplitude produced by the grating is a separate task.

We connect the field at one point to its time average and relative intensity. Run cells in order with NumPy, SciPy, Matplotlib, Plotly, and ipywidgets. The two explorers and calculation have independent settings. No earlier notebook needs to be run.

We retain a monochromatic scalar field, a homogeneous lossless medium, fixed measurement geometry, and $e^{-i\omega t}$. Field units, references, and plot scales remain fixed.

In [ ]:
# Find the shared wave_optics package in the project root.
from pathlib import Path
import sys
import importlib

_support_dir = next((base
    for base in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
    if (base / 'wave_optics' / '__init__.py').is_file()), None)
if _support_dir is None:
    raise FileNotFoundError('Open this notebook within the Grating_Physics folder.')
if str(_support_dir) not in sys.path:
    sys.path.insert(0, str(_support_dir))
# Relocate a cached package without changing the lesson's module reload order.
importlib.invalidate_caches()
if 'wave_optics' in sys.modules:
    sys.modules['wave_optics'].__path__[:] = [str(_support_dir / 'wave_optics')]

import importlib
import numpy as np
from IPython.display import display

# Rerunning setup releases any earlier copies of this notebook's views.
if 'amplitude_lessons' in globals():
    for old_lesson in amplitude_lessons.values():
        old_lesson.close()

# Import normally; restart the kernel after editing shared helpers.
import wave_optics.amplitude_view as amplitude_view
amplitude_lessons = {}

def show_amplitude_lesson(stage):
    if stage in amplitude_lessons:
        amplitude_lessons[stage].close()
    lesson = amplitude_view.AmplitudeIntensityExplorer(stage)
    amplitude_lessons[stage] = lesson
    display(lesson.ui)

## 1. The field at one observation point

For our selected order,
$$u_m(\mathbf r,t)=A_m\cos(\mathbf k_m\cdot\mathbf r+\phi_m-\omega t).$$
At a fixed point $P$, write $A=A_m$ and $\phi=\mathbf k_m\cdot\mathbf r_P+\phi_m$:
$$u(P,t)=A\cos(\phi-\omega t),\qquad U(P)=Ae^{i\phi}.$$

Here $A\geq0$ sets the extrema $\pm A$; $\phi$ sets the cycle's alignment with the time origin. The real field can be negative or cross zero while its amplitude stays nonzero.

An ordinary optical detector averages over many carrier cycles. The two panels below distinguish the **signed field** from its **square**. For one complete period $T$,
$$\langle u\rangle_T=0,\qquad
\langle u^2\rangle_T=A^2\langle\cos^2(\phi-\omega t)\rangle_T=\frac{A^2}{2}.$$
Angle brackets mean a time average. Squaring first prevents the positive and negative field readings from cancelling. A phase shift moves the cycle without changing its mean square.

**Try:** start with $A=1$, phase $0^\circ$. Change only phase to $90^\circ$, then change amplitude to 2. Which average changes? The horizontal axis is model time $t/T$; these curves are not camera frames.

In [ ]:
show_amplitude_lesson('average')


:::{dropdown} What to observe

At $A=1$, the mean square is $1/2$ for either phase, even though the field at $t=0$ changes from 1 to 0. At $A=2$, the mean square becomes 2. The mean signed field stays zero in every case.

:::

## 2. Convert the mean square into recorded intensity

For this travelling wave at fixed medium and detector geometry,
$$I=K\langle u^2\rangle_T=K\frac{A^2}{2}.$$
The positive constant $K$ accounts for the physical field units and the fixed geometry. Compare with a reference wave in the same direction and medium, with amplitude $A_{\rm ref}=1$ field unit:
$$\boxed{\frac{I}{I_{\rm ref}}
=\frac{KA^2/2}{KA_{\rm ref}^2/2}
=\frac{A^2}{A_{\rm ref}^2}
=\frac{|U|^2}{A_{\rm ref}^2}.}$$

This is the squared-magnitude rule previewed in Notebook 0. The factor $1/2$ cancels in the ratio. Our plots show dimensionless **relative intensity**; absolute intensity in $\mathrm{W/m^2}$ requires the physical conversion.

**Predict and try:** change amplitude from 1 to 0.5, then to 2. The relative intensities should be 1, 0.25, and 4. The reference and the graph scales remain fixed.

In [ ]:
show_amplitude_lesson('intensity')


### Small calculation 1: can the starting field predict brightness?

The code compares three phases at the same amplitude. **Predict:** which rows have the same intensity? Then change `A` from 1 to 2 and rerun. These values are independent of the sliders.

`np.mean(u**2)` averages squared field samples over one cycle, without duplicating the endpoint.

In [ ]:
A = 1.0  # field units; try 2.0
A_ref = 1.0  # keep this reference fixed
phase_deg = np.array([0.0, 90.0, 180.0])
tau = np.linspace(0.0, 1.0, 400, endpoint=False)
reference_mean_square = A_ref**2 / 2
print('phase (degrees)    u(0)    mean(u²)    I/I_ref')
for phase in phase_deg:
    u = A * np.cos(np.deg2rad(phase) - 2 * np.pi * tau)
    mean_square = np.mean(u**2)
    relative_intensity = mean_square / reference_mean_square
    print(f'{phase:8.0f}          {u[0]:6.3f}     {mean_square:6.3f}      {relative_intensity:6.3f}')

:::{dropdown} Answer

At $A=1$, the starting fields are $1,0,-1$, but every row has mean square ${0.5}$ and relative intensity 1. At $A=2$, the starting fields double and every relative intensity becomes 4. One instantaneous field reading does not determine brightness.

:::

### Read the measurement in reverse

For background-subtracted counts $S$ from a linear, unsaturated detector, keep the sampled area, exposure, wavelength, detector settings, and beam geometry the same:
$$\frac{S}{S_{\rm ref}}=\frac{I}{I_{\rm ref}}
\quad\Longrightarrow\quad
\frac{A}{A_{\rm ref}}=\sqrt{\frac{S}{S_{\rm ref}}}.$$
A change from 100 to 400 counts means four times the intensity and twice the field amplitude. These counts give no phase offset: all three phases in the calculation produce the same signal.

A real pixel samples a finite area even when our ideal field has no transverse edge. Uniform intensity gives a finite signal from that area; it is not a prediction of the total power of an infinite plane wave. The simple amplitude inference assumes that the sampled beam profile is unchanged. Moving a finite beam edge across the pixel can change counts without a uniform amplitude scaling.
- This uses the fixed-geometry convention in Goodman, §4.1.1, cited below. 

## Checkpoint

1. Why is the mean signed field zero while the mean square is nonzero?
2. What happens to relative intensity if amplitude doubles?
3. Can an intensity ratio determine the phase offset?

:::{dropdown} Answers

Positive and negative field readings cancel; their squares do not. Doubling amplitude multiplies intensity by four. A single-wave intensity ratio gives an amplitude ratio, while phase requires additional information.

:::

Continue to [6.2: one order across a detector](6_2_One_Order_Across_a_Detector.ipynb).

## Worked-out examples

These optional scripts show the numerical implementation of the topics in this notebook.
Use the complete [scripts download](Downloads.md) to keep the plotting helper and
`wave_optics` package with the examples. Edit **INPUT PARAMETERS**, then run the file
or execute its `# %%` sections in order. Read **CHECK AND INTERPRET** with the plots.

### 1. Time averaging and relative intensity

**Script:** [6_1_01_finite_exposure_average.py](Wave_solver_py_scripts/scripts/chapter_6/6_1_01_finite_exposure_average.py)

Integrate the real-field square over a finite detector exposure.

### 2. Convert mean square into relative recorded intensity

**Script:** [6_1_02_amplitude_intensity_scaling.py](Wave_solver_py_scripts/scripts/chapter_6/6_1_02_amplitude_intensity_scaling.py)

Verify the square-law relation with a fixed reference.

### 3. Numerically average one optical field

**Script:** [6_1_03_time_quadrature_convergence.py](Wave_solver_py_scripts/scripts/chapter_6/6_1_03_time_quadrature_convergence.py)

Check detector time-sampling convergence.

### 4. Read the measurement in reverse

**Script:** [6_1_04_intensity_loses_phase.py](Wave_solver_py_scripts/scripts/chapter_6/6_1_04_intensity_loses_phase.py)

Show that phase-distinct fields can have equal recorded intensity.




(refs-6-1)=

## References

(ref-6-1-openstax-energy)=

[1] OpenStax, “16.3 Energy Carried by Electromagnetic Waves,” *University Physics*, vol. 2. [Online]. Available: [OpenStax, University Physics, §16.3](https://openstax.org/books/university-physics-volume-2/pages/16-3-energy-carried-by-electromagnetic-waves).

## Further Reading

(ref-6-1-gbur-visser)=

- G. Gbur and T. D. Visser, “The Structure of Partially Coherent Fields,” *Progress in Optics*, vol. 55, pp. 285–341, 2010. [Online]. Available: [PDF](https://www.nat.vu.nl/~tvisser/PinO.pdf). doi: 10.1016/B978-0-444-53705-8.00005-9.

  **Reading notes:** **Access:** Free author-hosted review chapter; copyright retained. **Read for:** Read Sections 2–3: correlation, interference and visibility.

(ref-6-1-mansuripur)=

- M. Mansuripur, “A Tutorial on the Classical Theories of Electromagnetic Scattering and Diffraction,” *Nanophotonics*, vol. 10, pp. 315–342, 2021; arXiv version submitted 2020. [Online]. Available: [preprint](https://arxiv.org/abs/2009.06706).

  **Reading notes:** **Access:** Free arXiv manuscript; not public domain. **Read for:** Optional advanced reading: scalar diffraction and its electromagnetic assumptions.

## Reference notes

No earlier notebook must be executed. Rerunning setup closes earlier views; rerunning a display cell replaces that view. Saved outputs and widget state are clear. To release this lesson’s views when finished:

```python
for lesson in amplitude_lessons.values():
    lesson.close()
```

Freely accessible works are not necessarily in the public domain.

**Reading level:** The reviews extend beyond this lesson. Focus on averaged field products and intensity; their scattering and statistical developments are optional.



## Course connections

[Chapter 0](0_0_Foundations_of_Scalar_Optics.ipynb) and [Chapter 1](1_0_Plane_Waves_and_Wavevectors.ipynb): complex-field notation, phase, and wavevectors. [OpenStax, *University Physics*, §16.3](#ref-6-1-openstax-energy) [\[1\]](#ref-6-1-openstax-energy), Eqs. (16.30)–(16.31): cycle averaging and the quadratic relation between travelling-wave field amplitude and intensity. Goodman, *Introduction to Fourier Optics*, second edition, §4.1.1, Eq. (4-5), printed p. 65: scalar squared-magnitude intensity convention.
